# 00 — Generate simulation examples from restricted HTS (Transparency only)

Important note:
- The original HTS microdata are NOT included in this GitHub repository.
- This notebook documents how the exported simulation datasets were generated.
- The runnable pipeline operates on the exported CSV files saved under `path_od_sim`.

# Paths, seed, optional Colab mount

In [30]:
from pathlib import Path
import os
import random
import numpy as np
import pandas as pd
from typing import Tuple
import warnings
warnings.filterwarnings('ignore')

# Reproducibility (for sampling)
SEED = int(os.getenv("PROJECT_SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)

# Optional: Colab Drive mount (only needed if you run in Colab)
IN_COLAB = "COLAB_GPU" in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Author default (can override via environment variable OD_SIM_PATH)
DEFAULT_SIM_PATH = "/content/drive/MyDrive/Colab Notebooks/Workspace/od_matrix_generation/od_sim_data/"
path_od_sim = Path(os.getenv("OD_SIM_PATH", DEFAULT_SIM_PATH))

print("SEED =", SEED)
print("path_od_sim =", path_od_sim)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SEED = 42
path_od_sim = /content/drive/MyDrive/Colab Notebooks/Workspace/od_matrix_generation/od_simulation


# Helper functions (generation)

In [31]:
def build_zone_prior(df: pd.DataFrame, mode: str, zone_col: str) -> pd.DataFrame:
    """
    Build zone-level prior by summing counts for a given mode.
    Expects df with columns: [TRAVEL_MODE, zone_col, COUNT]
    Returns: [zone_col, COUNT]
    """
    out = df[df["TRAVEL_MODE"] == mode][[zone_col, "COUNT"]].copy()
    out["COUNT"] = out.groupby(zone_col)["COUNT"].transform("sum")
    return out.drop_duplicates()


def build_mode_counts(df: pd.DataFrame, key_od: list[str], coord_cols: list[str]) -> pd.DataFrame:
    """
    Aggregate trip-level HTS into unique OD rows with mode counts (PT/WK/VH),
    and attach coordinates (one deterministic row per OD).
    """
    tmp = df.loc[df["TRAVEL_MODE"].isin(["PT", "WK", "VH"]), key_od + ["TRAVEL_MODE"]]

    counts = (
        tmp.groupby(key_od + ["TRAVEL_MODE"])
           .size()
           .unstack("TRAVEL_MODE", fill_value=0)
           .reset_index()
           .rename(columns={"PT": "PT_COUNT", "WK": "WK_COUNT", "VH": "VH_COUNT"})
    )

    for col in ["PT_COUNT", "WK_COUNT", "VH_COUNT"]:
        if col not in counts.columns:
            counts[col] = 0

    # Deterministic coordinate attachment: first non-null coordinate row per OD
    coords = (
        df[key_od + coord_cols]
        .dropna(subset=coord_cols)
        .drop_duplicates(subset=key_od)
    )

    out = counts.merge(coords, on=key_od, how="left")
    return out.drop_duplicates(subset=key_od).reset_index(drop=True)


def compute_distance_km(df: pd.DataFrame) -> np.ndarray:
    """Haversine distance in km using ORIGIN_* and DESTINATION_* lon/lat columns."""
    lon1 = np.radians(df["ORIGIN_SUBZONE_X"].to_numpy(float))
    lon2 = np.radians(df["DESTINATION_SUBZONE_X"].to_numpy(float))
    lat1 = np.radians(df["ORIGIN_SUBZONE_Y"].to_numpy(float))
    lat2 = np.radians(df["DESTINATION_SUBZONE_Y"].to_numpy(float))

    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c


def destination_land_use_mix(
    hts_df: pd.DataFrame,
    *,
    dest_col: str = "DESTINATION_SUBZONE",
    purpose_col: str = "TRIP_PURPOSE",
    purposes: Tuple[str, ...] = ("H", "W", "O"),
    prefix: str = "LU",
) -> pd.DataFrame:
    """
    Destination land-use mix proxies as HTS destination-purpose shares.
    Output: [DESTINATION_SUBZONE, LU_H, LU_W, LU_O]
    """
    df = hts_df[[dest_col, purpose_col]]
    df = df[df[purpose_col].isin(purposes)].copy()

    counts = (
        df.groupby([dest_col, purpose_col], as_index=False)
          .size()
          .rename(columns={"size": "N"})
    )

    total_n = float(counts["N"].sum())
    if total_n <= 0:
        out = pd.DataFrame({dest_col: pd.unique(df[dest_col].dropna())})
        for p in purposes:
            out[f"{prefix}_{p}"] = 0.0
        return out[[dest_col] + [f"{prefix}_{p}" for p in purposes]]

    counts["Prob"] = counts["N"] / total_n
    denom = counts.groupby(dest_col)["Prob"].transform("sum")
    counts["Prob"] = counts["Prob"] / denom

    wide = (
        counts.pivot(index=dest_col, columns=purpose_col, values="Prob")
              .fillna(0.0)
              .reset_index()
    )

    rename_map = {p: f"{prefix}_{p}" for p in purposes if p in wide.columns}
    wide.rename(columns=rename_map, inplace=True)

    for p in purposes:
        col = f"{prefix}_{p}"
        if col not in wide.columns:
            wide[col] = 0.0

    return wide[[dest_col] + [f"{prefix}_{p}" for p in purposes]]



# Main generation logic (requires private HTS)

In [34]:
# -----------------------------
# CONFIG
# -----------------------------
KEY_OD = ["ORIGIN_SUBZONE", "DESTINATION_SUBZONE"]
COORDS = ["ORIGIN_SUBZONE_X", "ORIGIN_SUBZONE_Y", "DESTINATION_SUBZONE_X", "DESTINATION_SUBZONE_Y"]
reg = "sgp"  # "seoul" or "sgp"

# -----------------------------
# INPUT (private HTS) — NOT PROVIDED
# -----------------------------
# You must have this file locally/Drive to run notebook 00:
full_hts = pd.read_csv(path_od_sim / f"data_hts_{reg}.csv")

'''raise RuntimeError(
    "Notebook 00 is for transparency only. "
    "It requires restricted HTS microdata (data_hts_{reg}.csv), which is not provided in the GitHub repo."
)'''

full_hts = pd.read_csv(path_od_sim / f"data_hts_{reg}.csv")
full_hts = full_hts.drop(columns=["ID", "TRIP_STARTTIME"], errors="ignore")
full_hts["COUNT"] = 1

#full_hts = full_hts.sample(frac=0.7, random_state=SEED)

dest_lu_mix = destination_land_use_mix(full_hts, dest_col="DESTINATION_SUBZONE", purpose_col="TRIP_PURPOSE")

true_df = build_mode_counts(full_hts, KEY_OD, COORDS)
true_df["COUNT"] = true_df["PT_COUNT"] + true_df["WK_COUNT"] + true_df["VH_COUNT"]
true_df = true_df[true_df["PT_COUNT"] > 0].copy()

# Restrict HTS to OD support
full_hts = full_hts.merge(true_df[KEY_OD].drop_duplicates(), on=KEY_OD, how="inner")

# Priors
prior_origin_vh = build_zone_prior(full_hts, "VH", "ORIGIN_SUBZONE")
prior_origin_wk = build_zone_prior(full_hts, "WK", "ORIGIN_SUBZONE")
prior_destination_vh = build_zone_prior(full_hts, "VH", "DESTINATION_SUBZONE")
prior_destination_wk = build_zone_prior(full_hts, "WK", "DESTINATION_SUBZONE")

# Smart-card proxy: PT only + distance
sc_df = true_df[KEY_OD + COORDS + ["PT_COUNT"]].rename(columns={"PT_COUNT": "COUNT"}).copy()
sc_df["TRIP_DISTANCE"] = compute_distance_km(sc_df)
sc_df = sc_df.drop(columns=COORDS)
true_df = true_df.drop(columns=COORDS)

# Save exported simulation inputs
dest_lu_mix.to_csv(path_od_sim / f"data_sim_{reg}_destination_lu_mix.csv", index=False)
prior_origin_vh.to_csv(path_od_sim / f"data_sim_{reg}_prior_origin_vh.csv", index=False)
prior_origin_wk.to_csv(path_od_sim / f"data_sim_{reg}_prior_origin_wk.csv", index=False)
prior_destination_vh.to_csv(path_od_sim / f"data_sim_{reg}_prior_destination_vh.csv", index=False)
prior_destination_wk.to_csv(path_od_sim / f"data_sim_{reg}_prior_destination_wk.csv", index=False)
sc_df.to_csv(path_od_sim / f"data_sim_{reg}_sc.csv", index=False)
true_df.to_csv(path_od_sim / f"data_sim_{reg}_true.csv", index=False)

print(dest_lu_mix.info())
print(prior_origin_vh.info())
print(sc_df.info())
print(true_df.info())

# Generate 5 HTS samples (20% each)
for i in range(1, 6):
    hts_df = full_hts.sample(frac=0.2, random_state=SEED + i).copy()
    hts_df["TRIP_DISTANCE"] = compute_distance_km(hts_df)
    hts_df = hts_df.drop(columns=COORDS + ["TRIP_PURPOSE"], errors="ignore")
    hts_df.to_csv(path_od_sim / f"data_sim_{reg}_hts_v{i}.csv", index=False)
    print(hts_df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   DESTINATION_SUBZONE  299 non-null    object 
 1   LU_H                 299 non-null    float64
 2   LU_W                 299 non-null    float64
 3   LU_O                 299 non-null    float64
dtypes: float64(3), object(1)
memory usage: 9.5+ KB
None
<class 'pandas.core.frame.DataFrame'>
Index: 267 entries, 7 to 43468
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   ORIGIN_SUBZONE  267 non-null    object
 1   COUNT           267 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 6.3+ KB
None
<class 'pandas.core.frame.DataFrame'>
Index: 15328 entries, 0 to 21191
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0  